# 02 · Text Splitting (Data Transformation)

After loading documents, we split them into **smaller chunks**.

> RAG pipeline: Load ➜ **Split** ➜ Embed ➜ Store ➜ Retrieve

### Why split?
- LLMs have a limited **context window**, so we can't send a whole book.
- Embedding models have a **maximum input length**.
- Smaller, focused chunks give **more accurate retrieval**: the search returns only the relevant part, not a whole 15-page PDF.

### Key parameters
| Parameter | Meaning |
|---|---|
| `chunk_size` | Maximum size of each chunk (counted in **characters** by default, using `len`) |
| `chunk_overlap` | **Maximum** number of characters shared between consecutive chunks, so context isn't lost at chunk boundaries |

### What this notebook covers
1. `RecursiveCharacterTextSplitter`: general-purpose text (recommended default)
2. `HTMLHeaderTextSplitter`: HTML, split by headings
3. `RecursiveJsonSplitter`: JSON data

## Load a PDF to split
Same `PyPDFLoader` as in `01-DataIngestion.ipynb`, giving one `Document` per page.

In [6]:
## Reading a PDF file
from langchain_community.document_loaders import PyPDFLoader
pdfLoader =  PyPDFLoader('Attention.pdf')
docs = pdfLoader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszk

#### How to recursively split text by characters

## 1. RecursiveCharacterTextSplitter

The **recommended splitter for general text**. It tries separators in this order:

```
"\n\n"  ➜  "\n"  ➜  " "  ➜  ""
paragraph   line     word    character
```

It only moves to the next (smaller) separator when a piece is still bigger than `chunk_size`. This keeps paragraphs, then sentences, then words together as long as possible.

`split_documents(docs)` takes `Document`s and **copies the metadata** (source, page number, …) onto every chunk.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap=50)
final_documents = text_splitter.split_documents(docs)
final_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu'),
 Document(metadata={'producer': 'pd

### Checking the chunk overlap

⚠️ **How overlap really works:** `chunk_overlap` is a **maximum, not a guarantee**.
- The overlap is built from **whole pieces** (paragraphs, lines or words) at the end of the previous chunk, and only if they fit within `chunk_overlap` characters.
- If a chunk ends at a paragraph break (`\n\n`), or the last line is longer than `chunk_overlap`, **there is no overlap**.

That is why chunks 3 and 4 below share no text: they were split on a line break, and the last line of chunk 3 is longer than 50 characters.

In [8]:
print(final_documents[3])
print(final_documents[4])

page_content='our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started' metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
page_content='the effort to evaluate this idea. Ashish, with Illia, designed and implemente

## 2. Splitting raw text with `create_documents`
Load `speech.txt` again with `TextLoader`.

In [9]:
## Text loader

from langchain_community.document_loaders import TextLoader
loader = TextLoader('speech.txt')
docs = loader.load()
docs


[Document(metadata={'source': 'speech.txt'}, page_content="\nArtificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.\n\nArtificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.\n\nOne of the most important concepts in AI is Machine Learning. Machine Learning is a technique that allows computers to learn from data without being explicitly programmed for every task. For example, email spam filters use machine learning to identify unwanted emails based on previously observed patterns.\n\nAnother important concept is Deep Learning. Deep Learning is a subset of Machine Learning that uses artificial neural networks inspired by the human 

Read the same file as a **plain Python string**. `create_documents()` expects a list of strings, not `Document` objects.

In [11]:
speech = " "
with open('speech.txt') as f:
    speech = f.read()
speech


"\nArtificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.\n\nArtificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.\n\nOne of the most important concepts in AI is Machine Learning. Machine Learning is a technique that allows computers to learn from data without being explicitly programmed for every task. For example, email spam filters use machine learning to identify unwanted emails based on previously observed patterns.\n\nAnother important concept is Deep Learning. Deep Learning is a subset of Machine Learning that uses artificial neural networks inspired by the human brain. These networks contain multiple layers that help co

`create_documents([text])` splits the strings and wraps each chunk in a `Document`. Metadata is empty unless you pass `metadatas=[...]`.
Use `split_text(text)` instead if you just want a `list[str]`.

With `chunk_size=100` and `chunk_overlap=10`, the overlap is easy to see in the output, for example:
- chunk 1 ends with `"... and some of its"`
- chunk 2 starts with `"of its important concepts."`

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
                                                chunk_size = 100,
                                                chunk_overlap = 10
)

text = text_splitter.create_documents([speech])

print(text)

[Document(metadata={}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!'), Document(metadata={}, page_content='Today, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its'), Document(metadata={}, page_content='of its important concepts.'), Document(metadata={}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and'), Document(metadata={}, page_content='and software capable of performing tasks that normally require human intelligence. These tasks'), Document(metadata={}, page_content='tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'), Document(metadata={}, page_content='One of the most important concepts in AI is Machine Learning. Machine Learning is a technique that'), Document(metadata={}, page_content='that allows computers to learn from data without being explicitly programmed for every task. For'), Doc

### Inspecting individual chunks

In [15]:
print(text[0])


page_content='Artificial Intelligence: Concepts and Applications

Good morning everyone!'


In [16]:
print(text[1])

page_content='Today, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its'


In [17]:
print(text[2])

page_content='of its important concepts.'


In [18]:
print(text[3])

page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and'


In [19]:
print(text[4])

page_content='and software capable of performing tasks that normally require human intelligence. These tasks'


## 3. HTMLHeaderTextSplitter

A **structure-aware** splitter for HTML. It splits the page at the header tags you choose (`<h1>`, `<h2>`, …) and stores the headers each chunk sits under in its **metadata**.

Useful for documentation websites, where the heading tells you what the section is about. You can also split a live page directly with `split_text_from_url(url)`.

In [ ]:
### HTML text splitter

In [21]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_string = """
<!DOCTYPE html>
<html>
<head>
    <title>My First Web Page</title>
</head>
<body>

    <!-- Heading Tags -->
    <h1>Welcome to HTML</h1>
    <h2>This is a Heading 2</h2>
    <h3>This is a Heading 3</h3>

    <!-- Paragraph Tag -->
    <p>This is a sample paragraph in HTML.</p>

    <!-- Text Formatting Tags -->
    <b>Bold Text</b><br>
    <strong>Important Text</strong><br>
    <i>Italic Text</i><br>
    <u>Underlined Text</u><br>
    <mark>Highlighted Text</mark><br>
    <del>Deleted Text</del><br>

    <!-- Link Tag -->
    <a href="https://www.google.com">Visit Google</a>

    <br><br>

    <!-- Image Tag -->
    <img src="image.jpg" alt="Sample Image" width="200">

    <!-- List Tags -->
    <h2>Ordered List</h2>
    <ol>
        <li>HTML</li>
        <li>CSS</li>
        <li>JavaScript</li>
    </ol>

    <h2>Unordered List</h2>
    <ul>
        <li>Apple</li>
        <li>Mango</li>
        <li>Orange</li>
    </ul>

    <!-- Table Tags -->
    <table border="1">
        <tr>
            <th>Name</th>
            <th>Age</th>
        </tr>
        <tr>
            <td>John</td>
            <td>25</td>
        </tr>
        <tr>
            <td>David</td>
            <td>30</td>
        </tr>
    </table>

    <!-- Form Tags -->
    <h2>Registration Form</h2>
    <form>
        <label>Name:</label>
        <input type="text" placeholder="Enter your name">
        <br><br>

        <label>Email:</label>
        <input type="email" placeholder="Enter your email">
        <br><br>

        <button type="submit">Submit</button>
    </form>

</body>
</html>
 """

`headers_to_split_on` is a list of `(html_tag, metadata_key)` pairs.
Each resulting `Document` has the text of one section, with metadata like `{'Header 1': 'Welcome to HTML', 'Header 2': 'Ordered List'}`.

In [22]:
headers_to_split_on = [
    ("h1","Header 1"),
    ("h2","Header 2"),
    ("h3","Header 3")

]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)
html_header_splits

[Document(metadata={}, page_content='Heading Tags Paragraph Tag Text Formatting Tags Link Tag Image Tag List Tags Table Tags Form Tags'),
 Document(metadata={'Header 1': 'Welcome to HTML'}, page_content='Welcome to HTML'),
 Document(metadata={'Header 1': 'Welcome to HTML', 'Header 2': 'This is a Heading 2'}, page_content='This is a Heading 2'),
 Document(metadata={'Header 1': 'Welcome to HTML', 'Header 2': 'This is a Heading 2', 'Header 3': 'This is a Heading 3'}, page_content='This is a Heading 3'),
 Document(metadata={'Header 1': 'Welcome to HTML', 'Header 2': 'This is a Heading 2', 'Header 3': 'This is a Heading 3'}, page_content='This is a sample paragraph in HTML.  \nBold Text  \nImportant Text  \nItalic Text  \nUnderlined Text  \nHighlighted Text  \nDeleted Text  \nVisit Google'),
 Document(metadata={'Header 1': 'Welcome to HTML', 'Header 2': 'Ordered List'}, page_content='Ordered List'),
 Document(metadata={'Header 1': 'Welcome to HTML', 'Header 2': 'Ordered List'}, page_content

## 4. RecursiveJsonSplitter

Fetch a sample JSON document: the LangSmith OpenAPI specification. *(Needs an internet connection.)*

In [23]:
## JSON splitter

import json 
import requests

json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()

In [24]:
json_data

{'openapi': '3.1.0',
 'info': {'title': 'LangSmith',
  'description': 'The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n',
  'version': '0.1.0'},
 'paths': {'/api/v1/info/health': {'get': {'tags': ['info'],
    'summary': 'Get Health Info',
    'description': 'Get health information about the current deployment of LangSmith.',
    'operationId': 'get_health_info_api_v1_info_health_get',
    'responses': {'200': {'description': 'Successful Response',
      'content': {'application/json': {'schema': {'$ref': '#/components/schemas/HealthInfoGetResponse'}}}}},
    'x-public': True}},
  '/api/v1/sessions/{session_id}/dashboard': {'post': {'tags': ['tracer-sessions'],
    'summary': 'Get Tracing Project Prebuilt 

`RecursiveJsonSplitter` walks through nested JSON and splits it into **smaller JSON objects**, keeping the nested key path so every chunk is still valid, meaningful JSON.

- `max_chunk_size` is the target maximum size of each chunk in characters. A single value larger than the limit cannot be split further.
- `split_json()` returns a `list[dict]`, `split_text()` returns JSON strings, and `create_documents()` returns `Document`s.

> `max_chunk_size=30` is deliberately tiny so the splitting is easy to see. Real projects typically use 300–2000.

In [25]:
from langchain_text_splitters import RecursiveJsonSplitter
json_splitter = RecursiveJsonSplitter(max_chunk_size=30)
jsonn_chunks =  json_splitter.split_json(json_data)


In [26]:
jsonn_chunks

[{'openapi': '3.1.0', 'info': {'title': 'LangSmith'}},
 {'info': {'description': 'The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n'}},
 {'info': {'version': '0.1.0'},
  'paths': {'/api/v1/info/health': {'get': {'tags': ['info']}}}},
 {'paths': {'/api/v1/info/health': {'get': {'summary': 'Get Health Info'}}}},
 {'paths': {'/api/v1/info/health': {'get': {'description': 'Get health information about the current deployment of LangSmith.'}}}},
 {'paths': {'/api/v1/info/health': {'get': {'operationId': 'get_health_info_api_v1_info_health_get'}}}},
 {'paths': {'/api/v1/info/health': {'get': {'responses': {'200': {'description': 'Successful Response'}}}}}},
 {'paths': {'/api/v1/info/health': {'get': {'responses': 

Print the first 3 chunks:

In [27]:
for chunk in jsonn_chunks[:3]:
    print(chunk)

{'openapi': '3.1.0', 'info': {'title': 'LangSmith'}}
{'info': {'description': 'The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n'}}
{'info': {'version': '0.1.0'}, 'paths': {'/api/v1/info/health': {'get': {'tags': ['info']}}}}


## Summary

| Splitter | Best for | Splits on |
|---|---|---|
| `RecursiveCharacterTextSplitter` | Any plain text (default choice) | `\n\n` ➜ `\n` ➜ space ➜ character |
| `CharacterTextSplitter` | Simple text with one known separator | A single separator |
| `HTMLHeaderTextSplitter` | HTML pages / docs sites | Header tags (`h1`, `h2`, …) |
| `RecursiveJsonSplitter` | JSON / API responses | Nested JSON keys |

**Rules of thumb:** start with `chunk_size` 500–1000 and `chunk_overlap` about 10–20% of the chunk size, then tune based on retrieval quality.

➡️ **Next:** `03-Embeddings.ipynb`, which converts these chunks into vectors.